# SBA 930 — Reproducible Financial Forecast Evaluation
This notebook reproduces the 2025 holdout logic and forecast-error calculations for Claude Opus 5 (Model A) and Gemini 3.8 Flash (Model B).

In [1]:
from pathlib import Path
import csv, json, math, statistics
ROOT = Path('..')


## 1. Verify the 2025 holdout
Model-facing inputs stop at 2024. The 2025 row is reserved for evaluation.

In [2]:
def read_csv(p):
    with open(p, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))
train = read_csv(ROOT/'data/holdout/model_inputs_through_2024.csv')
holdout = read_csv(ROOT/'data/holdout/actuals_2025.csv')
assert all(int(r['Year']) <= 2024 for r in train)
assert len(holdout) == 1 and holdout[0]['Year'] == '2025'
holdout[0]


{'Year': '2025',
 'SP500 Year-end Close': '6845.50',
 'SP500 Return (%)': '16.387804061119',
 'WTI Annual Average ($/bbl)': '65.39',
 'WTI Change (%)': '-14.667884640480',
 'CPI December (SA)': '326.031',
 'Dec-to-Dec Inflation (%)': '2.653304114558'}

## 2. Recalculate forecast errors from saved model artifacts

The notebook loads each model's structured forecast artifact from `outputs/`; forecast values are not typed into the calculation cell.


In [3]:
actual_sp = float(holdout[0]['SP500 Year-end Close'])
actual_wti = float(holdout[0]['WTI Annual Average ($/bbl)'])
actual_inf = float(holdout[0]['Dec-to-Dec Inflation (%)'])

with open(ROOT/'outputs/claude_opus_5/forecast_summary.json', encoding='utf-8') as f:
    claude = json.load(f)
with open(ROOT/'outputs/gemini_3_8_flash/forecast_summary.json', encoding='utf-8') as f:
    gemini = json.load(f)

ape = lambda forecast, actual: abs(forecast-actual)/abs(actual)*100

forecast_specs = {
    'S&P 500': ('sp500_2025_year_end', actual_sp, 'pct'),
    'WTI': ('wti_2025_annual_average', actual_wti, 'pct'),
    'Inflation (pp)': ('inflation_2025_dec_to_dec_pct', actual_inf, 'pp'),
}

results = {}
for label, (key, actual, metric) in forecast_specs.items():
    claude_forecast = float(claude['forecasts'][key]['point_estimate'])
    gemini_forecast = float(gemini['forecasts'][key]['point_estimate'])
    if metric == 'pct':
        claude_error = ape(claude_forecast, actual)
        gemini_error = ape(gemini_forecast, actual)
    else:
        claude_error = abs(claude_forecast - actual)
        gemini_error = abs(gemini_forecast - actual)
    results[label] = {
        'Claude forecast': claude_forecast,
        'Claude error': claude_error,
        'Gemini forecast': gemini_forecast,
        'Gemini error': gemini_error,
    }

results


{'S&P 500': {'Claude forecast': 6550.0,
  'Claude error': 4.316704404353225,
  'Gemini forecast': 6100.0,
  'Gemini error': 10.89036593382514},
 'WTI': {'Claude forecast': 72.0,
  'Claude error': 10.10857929346995,
  'Gemini forecast': 82.0,
  'Gemini error': 25.40143752867411},
 'Inflation (pp)': {'Claude forecast': 2.8,
  'Claude error': 0.14669588544199996,
  'Gemini forecast': 3.2,
  'Gemini error': 0.5466958854420003}}

## 3. Supporting risk statistics

In [4]:
rows = read_csv(ROOT/'data/processed/financial_dataset.csv')
sp = [float(r['SP500 Return (%)']) for r in rows if r['SP500 Return (%)']]
wt = [float(r['WTI Change (%)']) for r in rows if r['WTI Change (%)']]
{'S&P return stdev (pp)': statistics.stdev(sp), 'WTI change stdev (pp)': statistics.stdev(wt)}


{'S&P return stdev (pp)': 19.243752979386574,
 'WTI change stdev (pp)': 39.73104149875813}

## 4. Reproduce everything from the command line
Run `python src/validate_project.py` from the repository root to regenerate and validate the core artifacts.